# MeChess 1/2: collect books, annotated games and explanations (CPU notebook)

**Run this notebook with the accelerator OFF (None).** Collecting and parsing text is CPU work: it does not use your 30 h/week GPU quota. The GPU notebook
(*MeChess 2/2*) then trains the language model on what this one produces.

What it gathers (all through the repository's `chessme books-learn`, which also runs on a laptop):
- 100+ public-domain chess books and periodicals from Project Gutenberg / Internet Archive: game lines (algebraic and descriptive notation) and concept counts;
- annotated games: the ChessGPT annotated archive, its larger annotated-PGN shards, and the CC0 chess-studies dataset: every comment, glyph (`! ? !! ?? !? ?!`, `± ∓ =`...) and the position it belongs to;
- explanations in prose: the chess Stack Exchange (questions and answers) and chess Wikipedia articles;
- Lichess opening names (CC0).

**How to run**
1. *Settings -> Internet -> On* (phone verification once), *Accelerator -> None*. Put your repository URL in `REPO_URL`.
2. Click **Save Version -> Save & Run All (Commit)**. A committed run keeps going without your browser and its output is kept; an interactive session can be lost.
3. Afterwards open the notebook's *Output* tab: that folder is what the GPU notebook takes as input (*Add Input -> Notebook output files*).

**Safety.** Step 1 (`books-preflight`) checks dependencies, disk and every download URL in about a minute and stops before anything long starts. The main step is resumable: rerunning it skips
what is finished. Terms: public-domain books only; annotated games and Stack Exchange / Wikipedia text keep their own licences (Stack Exchange and Wikipedia are CC BY-SA): research and learning use, do not redistribute.

In [ ]:
import os, subprocess, sys, pathlib, time

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here
OUT = "/kaggle/working/learn" if os.path.exists("/kaggle") else "learn_out"
T0 = time.time()

def sh(*args):
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    if "<you>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your repository first.")
    sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests")
CLI = [sys.executable, "-m", "chessme"]
print("ready in", os.getcwd())

## Step 1. Preflight (about a minute): stops here if anything is missing

In [ ]:
sh(*CLI, "books-preflight", "--out", OUT, "--min-free-gb", "4")

## Step 2. Collect everything (resumable; rerun the cell if the session was interrupted)

In [ ]:
sh(*CLI, "books-learn", "--out", OUT, "--log", f"{OUT}/learn.log")
print(f"elapsed {(time.time() - T0) / 60:.1f} min")

In [ ]:
print(pathlib.Path(OUT, "report.md").read_text())

## Optional: your own PDFs (private notebook only)
Upload PDFs as a *private* Kaggle dataset. A PDF with a text layer is read; a scan is reported as needing OCR. Never publish the outputs.

In [ ]:
if False:                 # set True for your own books
    sh(sys.executable, "-m", "pip", "-q", "install", "pypdf")
    sh(*CLI, "books-pdf", "/kaggle/input", "--out", f"{OUT}/private", "--log", f"{OUT}/pdf.log")

In [ ]:
# Keep the output small: the raw downloads are not needed by the GPU notebook
import shutil
for p in ("annotated/annotated_pgn_free.tar.gz", "annotated/chessgpt", "annotated/lichess_studies.csv", "annotated/others.csv", "prose/raw"):
    q = pathlib.Path(OUT, p)
    if q.is_dir():
        shutil.rmtree(q, ignore_errors=True)
    elif q.exists():
        q.unlink()
print(sum(f.stat().st_size for f in pathlib.Path(OUT).rglob("*") if f.is_file()) / 1e6, "MB of output kept in", OUT)